# TESLA Cavity

This notebook demonstrates a complete TESLA waveguide analysis workflow:

1. Geometry creation
2. Frequency-domain analysis
3. Model order reduction
4. Comparison with analytical solution

In [1]:
from utils.visualization import *
from geometry.importers import STEPImporter, TESLACavity
from solvers.frequency_domain import FrequencyDomainSolver
from rom.reduction import ModelOrderReduction
from analytical.rectangular_waveguide import RWGAnalytical
from analytical.cst_result import CSTResult
from geometry.assembly import Assembly
from ngsolve.webgui import Draw # must import Draw, otherwise may run into problems showing mesh
from utils.visualization import (
    plot_z_comparison, 
    plot_s_comparison, 
    plot_all_parameters,
    plot_eigenfrequencies,
    ParameterPlotter,
    EigenfrequencyPlotter
)
%matplotlib widget
plt.rcParams['figure.dpi'] = 100

## 1. Define Geometry

Create a rectangular waveguide with specified dimensions and mesh parameters.

In [2]:
# tesla = TESLACavity(r"tesla3cell.geo")
# tesla.save_step(r"./tesla3cell.step")

In [3]:
# 1. Load and prepare geometry
tesla = STEPImporter(r"./tesla3cell.step", auto_build=False)
tesla.add_splitting_plane_at_x(0.0577)
tesla.add_splitting_plane_at_x(-0.0577)
tesla.split()

tesla.show_split_preview()
tesla.build()
tesla.name_solids()

tesla.generate_mesh(maxh=0.05) # after naming solids, must generate mesh but avoid rebuilding

Named 3 solids, 4 ports
  External: port1, port4
  Internal: port2, port3
Displayed 3 solids with distinct colors
Displayed 2 splitting plane(s)


Named 3 solids, 4 ports
  External: port1, port4
  Internal: port2, port3


In [4]:
# # 1. Load and prepare geometry
# midcell = STEPImporter(r"./tesla_midcell.step", auto_build=False)
# endcell_l = STEPImporter(r"./tesla_endcell_l.step", auto_build=False)
# endcell_r = STEPImporter(r"./tesla_endcell_r.step", auto_build=False)
# midcell.build()
# endcell_l.build()
# endcell_r.build()

# tesla = Assembly(main_axis='X')
# tesla.add('endcell_l', endcell_l)
# tesla.add('midcell1', midcell, after="endcell_l")
# tesla.add('endcell_r', endcell_r, after='midcell1')

# tesla.inspect()
# tesla.build()
# # geo.name_solids()
# tesla.generate_mesh(maxh=0.01)
# # tesla.mesh.GetBoundaries()

In [5]:
tesla.show('mesh')

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.25…

In [ ]:
# # 3. FOM solve
fmin, fmax = 0.1, 5
fds = FrequencyDomainSolver(tesla, order=1)
fds.save(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\pillbox")
fds.assemble_matrices(nportmodes=3)
results = fds.solve(fmin, fmax, 30, store_snapshots=True)


Structure Topology
Type: Compound structure
Domains (3): ['cell_1', 'cell_2', 'cell_3']
Total Ports (4): ['port1', 'port3', 'port3', 'port4']
External Ports (2): ['port1', 'port4']
Internal Ports (2): ['port3', 'port3']

Domain-Port Mapping:
  cell_1: ['port1 (external, input)', 'port3 (internal)']
  cell_2: ['port3 (internal)', 'port3 (internal)']
  cell_3: ['port3 (internal)', 'port4 (external, output)']
Project 'C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\pillbox' already exists at C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\pillbox
Options:
 [L] Load existing data
 [R] Rerun (OVERWRITE existing data)
 [N] New name (Rename current project)


In [ ]:
# fds = FrequencyDomainSolver.load(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\rwg_split_project")

In [ ]:
fds.plot_port_mode('port1', 1)

In [ ]:
#import cst result
which = ['1(2)1(2)']
cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\tesla3cell')

fig, ax = fds.foms.plot_s(which)
fig, ax = cstresult.plot_s(which, ax=ax)
plt.show()

In [ ]:
# fds.plot_field(41)

In [ ]:
roms = fds.foms.reduce()
%time roms_concat = roms.concatenate()
print()
%time result = roms_concat.solve(fmin, fmax, 1000)
print()
%time roms_concat_rom = roms_concat.reduce()
print()
%time result = roms_concat_rom.solve(fmin, fmax, 1000)
print()

In [ ]:
cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\tesla3cell')
which = ['1(2)1(3)']

fig, ax = cstresult.plot_s(which, linewidth=2)
# fig, ax = fds.foms.plot_s(['1(1)2(1)'], ax=ax,  marker='o', mfc='none', lw=0)
# roms.plot_s(['1(1)2(1)'], ax=ax)
fig, ax = roms_concat.plot_s(which, ax=ax)
roms_concat_rom.plot_s(which, ax=ax)
plt.show()

In [ ]:
fds.save(r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\tesla3cell")

In [ ]:
fds.foms.plot_residual('residual')
plt.show()

In [ ]:
# fig, ax = plot_eigenfrequencies([fds, roms_concat], analytical=cstresult,
#                       labels=['FOM', 'ROM'], n_modes=50)
# ax.set_ylim(0.1, 5)
# ax.set_xlim(0.1, 5)
# plt.show()

In [ ]:
fds.plot_port_mode('port3', 2)

In [ ]:
# %%time

# from netgen.occ import *
# from ngsolve import *
# from ngsolve.webgui import Draw
# from ngsolve.bem import *

# kappa=10
# order=4

# screen = WorkPlane(Axes( (0,0,0), Z, X)).RectangleC(15,15).Face()

# sp = Fuse(Sphere( (0,0,0), pi).faces)
# screen.faces.name="screen"
# sp.faces.name="sphere"
# shape = Compound([screen,sp])

# mesh = shape.GenerateMesh(maxh=5/kappa).Curve(order)
# Draw (mesh);

# fes_sphere = Compress(SurfaceL2(mesh, order=order, complex=True, definedon=mesh.Boundaries("sphere")))
# u,v = fes_sphere.TnT()
# fes_screen = Compress(SurfaceL2(mesh, order=order, dual_mapping=True, complex=True, definedon=mesh.Boundaries("screen")))
# print ("ndof_sphere = ", fes_sphere.ndof, "ndof_screen =", fes_screen.ndof)

# with TaskManager():
#     # V = HelmholtzSingleLayerPotentialOperator(fes_sphere, fes_sphere, kappa=kappa, intorder=10)
#     # K = HelmholtzDoubleLayerPotentialOperator(fes_sphere, fes_sphere, kappa=kappa, intorder=10)
#     # C = HelmholtzCombinedFieldOperator(fes_sphere, fes_sphere, kappa=kappa, intorder=10)
#     C = HelmholtzCF(u*ds("sphere"), kappa)*v*ds
#     u,v  = fes_sphere.TnT()
#     Id = BilinearForm(u*v*ds).Assemble()

# lhs = 0.5 * Id.mat + C.mat
# source = exp(1j * kappa * x)
# rhs = LinearForm(-source*v*ds).Assemble()

# gfu = GridFunction(fes_sphere)
# pre = BilinearForm(u*v*ds, diagonal=True).Assemble().mat.Inverse()
# with TaskManager():
#     gfu.vec[:] = solvers.GMRes(A=lhs, b=rhs.vec, pre=pre, maxsteps=40, tol=1e-8)

In [ ]:
# project_name = fr'C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\tesla3cell'
# fds = FrequencyDomainSolver.load(project_name)